In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# 1. Load dataset
data = pd.read_csv(r"D:\DL\LP IV Updated Datasets\AutoEncoder\CreditCard\creditcard.csv")
data = data.dropna()

x = data.drop('Class', axis=1)
y = data['Class']

# 2. Scale features
x = StandardScaler().fit_transform(x)

# 3. Train only on normal transactions
x_train = x[y == 0]

# 4. Build Autoencoder
inp = Input(shape=(x.shape[1],))
enc = Dense(16, activation='relu')(inp)
enc = Dense(8, activation='relu')(enc)
dec = Dense(16, activation='relu')(enc)
out = Dense(x.shape[1], activation='linear')(dec)

autoencoder = Model(inp, out)

# 5. Compile model
autoencoder.compile(optimizer='adam', loss='mse')

# 6. Train model
autoencoder.fit(x_train, x_train, epochs=10, batch_size=256)

# 7. Calculate reconstruction error
reconstructed = autoencoder.predict(x)
error = np.mean((x - reconstructed) ** 2, axis=1)

# 8. Set threshold and detect fraud
threshold = np.percentile(error[y == 0], 95)
y_pred = (error > threshold).astype(int)

# 9. Evaluate model
print("Threshold:", threshold)
print("Fraud detected:", sum(y_pred))
print(classification_report(y, y_pred))

# 10. Plot reconstruction error
plt.hist(error, bins=50)
plt.axvline(threshold, color='red')
plt.xlabel('Reconstruction Error')
plt.ylabel('Transactions')
plt.show()
